

### **K-Means**

Partition data into *k* clusters where each point belongs to the cluster with the nearest mean (centroid).

<br>

<p align="center">
<img src="visualizations/k-means.png" width="600">
</p>


### **Algorithm Steps**

1. **Choose k** (number of clusters).
2. **Initialize**: Randomly select k data points as initial centroids.
3. **Assign**: Each point → closest centroid.
4. **Update**: Move centroids to the mean of their assigned points.
5. **Repeat** steps 3–4 until convergence (centroids stop changing or fixed iterations).


### **Objective Function**

Minimize total within-cluster variance:

$$
\text{argmin}_S \sum_{i=1}^{k} \sum_{x \in S_i} \|x - \mu_i\|^2
$$

---

### ✅ **Advantages**

* Simple and fast.
* Works well on spherical, similarly sized clusters.


### ⚠️ **Limitations**

* Must predefine k.
* Sensitive to initialization and outliers.
* Poor for non-convex or unequal-sized clusters.
* Requires feature scaling.

---

### 🔧 **Variants & Techniques**

* **K-means++**: Smart initialization.
* **Elbow method**: Helps choose k.
* **Silhouette score**: Measures clustering quality.


In [1]:
import numpy as np
from tqdm import tqdm
from coil20.coil20_utils import load_all_images, extract_images_tsne

import sys
import os
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(project_root)
from images.image_preprocessing import (
    extract_hog,
)

In [2]:
class KMeans:
    def __init__(self, k, max_iters=100, tol=1e-4):
        self.k = k
        self.max_iters = max_iters
        self.tol = tol  # tolerance

    def fit(self, X):
        n_samples, n_features = X.shape

        # Step 1: Randomly initialize centroids by picking k data points
        random_indices = np.random.choice(n_samples, self.k, replace=False)
        self.centroids = X[random_indices]

        for i in tqdm(range(self.max_iters), desc="KMeans"):
            # Step 2: Assign clusters (compute distance to centroids)
            distances = self._compute_distances(X)
            self.labels = np.argmin(distances, axis=1)

            # Step 3: Recalculate centroids
            new_centroids = np.array(
                [X[self.labels == j].mean(axis=0) for j in range(self.k)]
            )

            # Step 4: Check for convergence
            diff = np.linalg.norm(self.centroids - new_centroids, axis=1)
            if np.all(diff < self.tol):
                break

            self.centroids = new_centroids

    def predict(self, X):
        distances = self._compute_distances(X)
        return np.argmin(distances, axis=1)

    def _compute_distances(self, X):
        return np.linalg.norm(X[:, np.newaxis] - self.centroids, axis=2)

In [5]:
# Load data
X_images, y_labels = load_all_images()
X_tsne = extract_images_tsne(extract_hog(X_images), n_components=3)

# Train algorithm
kmeans = KMeans(k=20)
kmeans.fit(X_tsne)

# Evaluate
from sklearn.metrics import adjusted_rand_score

y_pred = kmeans.predict(X_tsne)

ari = adjusted_rand_score(y_labels, y_pred)
print(f"Adjusted Rand Index: {ari:.4f}")

KMeans:  20%|██        | 20/100 [00:00<00:00, 1503.98it/s]

Adjusted Rand Index: 0.7838
